# Execução da camada Analítica — ENEM por UF

Este notebook executa a construção da camada **Analítica** para as edições de **1998 a 2025**, utilizando os arquivos Gold da Unidade Federativa definida em `UF_EXECUCAO`.

### Regras de operação
- **Parametrização:** para processar outra UF, altere apenas `UF_EXECUCAO` na Seção 1.
- **Execução centralizada:** toda a construção dos produtos analíticos é realizada por `executar_camada_analitica()`, implementada em `enem_pipeline.analytic`.
- **Nove produtos:** são gerados os produtos analíticos definidos em `analytic.py`, além do catálogo e do manifesto da camada.
- **Integridade:** a rotina audita as fontes Gold, verifica as referências territoriais e valida os Parquets gravados.
- **Sem junção individual:** os produtos são agregados e não alteram os arquivos Gold.
- **Fail-fast:** a execução é interrompida caso falte uma fonte Gold ou ocorra uma falha no processamento.

> **Nota:** este notebook apenas configura, executa e resume o processo. Não replica as regras nem as funções já implementadas em `analytic.py`. As camadas Bronze, Staging, Silver e Gold não são modificadas.

## 1. Configuração e Importação de Módulos

Carrega as configurações do projeto, importa a rotina de execução Analítica, normaliza a UF selecionada e define os diretórios de origem e destino.

In [1]:
from _setup import *

from enem_pipeline.config import (
    PROJECT_ROOT,
    ANO_INICIAL,
    ANO_FINAL,
    GOLD_DIR,
    ANALYTIC_DIR,
    METADATA_DIR,
    normalizar_uf,
)

from enem_pipeline.analytic import executar_camada_analitica

from pathlib import Path
import pandas as pd

UF_EXECUCAO = "PA"
UF_EXECUCAO = normalizar_uf(UF_EXECUCAO)

DIRETORIO_GOLD_UF = GOLD_DIR / f"uf={UF_EXECUCAO}"
DIRETORIO_ANALYTIC_UF = ANALYTIC_DIR / f"uf={UF_EXECUCAO}"
MANIFESTO_GOLD = METADATA_DIR / f"manifesto_gold_{UF_EXECUCAO}.csv"
MANIFESTO_ANALYTIC = METADATA_DIR / f"manifesto_analytic_{UF_EXECUCAO}.csv"
CATALOGO_ANALYTIC = METADATA_DIR / f"catalogo_analytic_{UF_EXECUCAO}.csv"

print("Raiz do projeto:", PROJECT_ROOT)
print("UF selecionada:", UF_EXECUCAO)
print("Período:", f"{ANO_INICIAL}–{ANO_FINAL}")
print("Origem Gold:", DIRETORIO_GOLD_UF)
print("Destino Analytic:", DIRETORIO_ANALYTIC_UF)
print("Manifesto Gold:", MANIFESTO_GOLD)
print("Manifesto Analytic:", MANIFESTO_ANALYTIC)
print("Catálogo Analytic:", CATALOGO_ANALYTIC)

Raiz do projeto: /home/akel/PycharmProjects/Projeto_ENEM
UF selecionada: PA
Período: 1998–2025
Origem Gold: /home/akel/PycharmProjects/Projeto_ENEM/data/gold/uf=PA
Destino Analytic: /home/akel/PycharmProjects/Projeto_ENEM/data/analytic/uf=PA
Manifesto Gold: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/manifesto_gold_PA.csv
Manifesto Analytic: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/manifesto_analytic_PA.csv
Catálogo Analytic: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/catalogo_analytic_PA.csv


## 2. Conferência dos Parâmetros de Execução

Confere o período, a existência do módulo `analytic.py` e a disponibilidade do manifesto Gold. Verifica também se as 48 saídas Gold esperadas para o período completo estão registradas e se todos os arquivos de origem existem.

In [2]:
assert ANO_INICIAL == 1998
assert ANO_FINAL == 2025
assert len(UF_EXECUCAO) == 2
assert (PROJECT_ROOT / "src" / "enem_pipeline" / "analytic.py").exists()
assert MANIFESTO_GOLD.is_file(), f"Manifesto Gold não encontrado: {MANIFESTO_GOLD}"

manifesto_gold = pd.read_csv(
    MANIFESTO_GOLD,
    sep=";",
    dtype={"uf": "string"},
)

colunas_obrigatorias = {
    "ano", "base", "arquivo_gold", "tipo_referencia_territorial"
}
colunas_ausentes = colunas_obrigatorias - set(manifesto_gold.columns)
assert not colunas_ausentes, (
    f"Colunas ausentes no manifesto Gold: {sorted(colunas_ausentes)}"
)

manifesto_gold["ano"] = pd.to_numeric(
    manifesto_gold["ano"], errors="raise"
).astype(int)

manifesto_gold_periodo = manifesto_gold[
    manifesto_gold["ano"].between(ANO_INICIAL, ANO_FINAL)
].copy()

manifesto_gold_periodo["gold_existe"] = manifesto_gold_periodo[
    "arquivo_gold"
].map(lambda caminho: Path(str(caminho)).is_file())

print("UF:", UF_EXECUCAO)
print("Anos:", ANO_INICIAL, "a", ANO_FINAL)
print("Saídas Gold no manifesto:", len(manifesto_gold_periodo))
print("Arquivos Gold encontrados:", int(manifesto_gold_periodo["gold_existe"].sum()))

if not manifesto_gold_periodo["gold_existe"].all():
    display(
        manifesto_gold_periodo.loc[
            ~manifesto_gold_periodo["gold_existe"],
            ["ano", "base", "tipo_referencia_territorial", "arquivo_gold"],
        ]
    )
    raise FileNotFoundError(
        "Existem arquivos Gold ausentes. Corrija a camada Gold antes de executar a Analítica."
    )

assert len(manifesto_gold_periodo) == 48, (
    f"Quantidade inesperada de saídas Gold: {len(manifesto_gold_periodo)} "
    "(esperado: 48 para 1998–2025)."
)

print("Conferência das fontes Gold concluída.")

UF: PA
Anos: 1998 a 2025
Saídas Gold no manifesto: 48
Arquivos Gold encontrados: 48
Conferência das fontes Gold concluída.


## 3. Processamento Gold → Analítica

Executa `executar_camada_analitica()` para toda a série histórica da UF selecionada. A rotina constrói os nove produtos analíticos, grava os arquivos Parquet, o catálogo e o manifesto, além de retornar a auditoria das fontes Gold.

`gravar=True` efetiva a gravação dos produtos no diretório da UF.

In [3]:
try:
    resultado_analytic = executar_camada_analitica(
        manifesto_gold_path=MANIFESTO_GOLD,
        metadata_dir=METADATA_DIR,
        analytic_root=ANALYTIC_DIR,
        uf=UF_EXECUCAO,
        ano_inicial=ANO_INICIAL,
        ano_final=ANO_FINAL,
        gravar=True,
    )

except Exception as erro:
    print(
        f"FALHA na execução Analítica: "
        f"{type(erro).__name__}: {erro}"
    )
    print("Execução interrompida. Corrija a falha antes de continuar.")
    raise

print("\nExecução Analítica concluída.")
print(f"Fontes Gold auditadas: {len(resultado_analytic['fontes_auditadas'])}")
print(f"Produtos analíticos: {len(resultado_analytic['produtos'])}")
print(f"Tempo: {resultado_analytic['duracao_segundos']:.2f} s")
print("Diretório:", resultado_analytic["diretorio_analytic"])
print("Catálogo:", CATALOGO_ANALYTIC)
print("Manifesto:", MANIFESTO_ANALYTIC)


Execução Analítica concluída.
Fontes Gold auditadas: 48
Produtos analíticos: 9
Tempo: 33.93 s
Diretório: /home/akel/PycharmProjects/Projeto_ENEM/data/analytic/uf=PA
Catálogo: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/catalogo_analytic_PA.csv
Manifesto: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/manifesto_analytic_PA.csv


## 4. Resumo e Auditoria da Execução

Apresenta a auditoria das fontes Gold, o resumo dos nove produtos gerados e a situação dos arquivos Parquet, catálogo e manifesto.

In [4]:
df_fontes_analytic = resultado_analytic["fontes_auditadas"].copy()
df_catalogo_analytic = resultado_analytic["catalogo"].copy()
df_manifesto_analytic = resultado_analytic["manifesto"].copy()

df_catalogo_analytic["arquivo_existe"] = df_catalogo_analytic[
    "arquivo_analytic"
].map(lambda caminho: Path(str(caminho)).is_file())

print("FONTES GOLD AUDITADAS")
display(df_fontes_analytic)

print("\nPRODUTOS ANALÍTICOS")
display(df_catalogo_analytic)

print("\nMANIFESTO ANALÍTICO")
display(df_manifesto_analytic)

print("\nRESUMO")
print("Fontes Gold auditadas:", len(df_fontes_analytic))
print("Produtos catalogados:", len(df_catalogo_analytic))
print("Arquivos Analytic existentes:", int(df_catalogo_analytic["arquivo_existe"].sum()))
print("Linhas analíticas:", int(df_catalogo_analytic["linhas"].sum()))
print("Tempo de execução:", f"{resultado_analytic['duracao_segundos']:.2f} s")

FONTES GOLD AUDITADAS


,ano,base,referencia,arquivo_gold,colunas
0,1998,enem,RESIDENCIA,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,33
1,1999,enem,RESIDENCIA,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,33
2,2000,enem,RESIDENCIA,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,33
3,2001,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,33
4,2002,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,33
5,2003,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,33
6,2004,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,33
7,2005,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,33
8,2006,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,33
9,2007,enem,ESC,/home/akel/PycharmProjects/Projeto_ENEM/data/g...,33



PRODUTOS ANALÍTICOS


,produto,arquivo_analytic,granularidade,periodo,linhas,colunas,descricao,arquivo_existe
0,demografia_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,ano_municipio,1998-2025,6464,11,"Contagem de registros e distribuição por sexo,...",True
1,faixa_etaria_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,ano_municipio_faixa_etaria,1998-2025,63836,7,"Distribuição por faixa etária, município, ano ...",True
2,presenca_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,ano_municipio,1998-2025,6464,11,Presença/ausência geral quando TP_PRESENCA est...,True
3,desempenho_redacao_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,ano_municipio,1998-2025,6464,13,"Estatísticas de redação, com contagem de zeros...",True
4,desempenho_objetiva_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,ano_municipio,1998-2008,1454,13,"Estatísticas da nota objetiva, incluindo zeros...",True
5,desempenho_areas_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,ano_municipio,2009-2025,5010,41,"Estatísticas de CN, CH, LC e MT, com contagem ...",True
6,escola_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,ano_municipio,1998-2025,6464,17,Distribuição por dependência administrativa da...,True
7,desempenho_grupos_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,ano_municipio_grupo_disciplina,1998-2025,337214,16,"Desempenho por sexo, escola e faixa etária; em...",True
8,desempenho_sexo_faixa_etaria_municipal,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,ano_municipio_sexo_faixa_etaria_disciplina,2001-2023,451205,17,"Estatísticas de desempenho por município, sexo...",True



MANIFESTO ANALÍTICO


,camada,uf,produto,granularidade,periodo,arquivo_analytic,n_linhas,n_colunas,tamanho_mb
0,analytic,PA,demografia_municipal,ano_municipio,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,6464,11,0.123483
1,analytic,PA,faixa_etaria_municipal,ano_municipio_faixa_etaria,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,63836,7,0.142610
2,analytic,PA,presenca_municipal,ano_municipio,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,6464,11,0.060381
3,analytic,PA,desempenho_redacao_municipal,ano_municipio,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,6464,13,0.197515
4,analytic,PA,desempenho_objetiva_municipal,ano_municipio,1998-2008,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,1454,13,0.057270
5,analytic,PA,desempenho_areas_municipal,ano_municipio,2009-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,5010,41,0.693615
6,analytic,PA,escola_municipal,ano_municipio,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,6464,17,0.155934
7,analytic,PA,desempenho_grupos_municipal,ano_municipio_grupo_disciplina,1998-2025,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,337214,16,8.590934
8,analytic,PA,desempenho_sexo_faixa_etaria_municipal,ano_municipio_sexo_faixa_etaria_disciplina,2001-2023,/home/akel/PycharmProjects/Projeto_ENEM/data/a...,451205,17,10.113083



RESUMO
Fontes Gold auditadas: 48
Produtos catalogados: 9
Arquivos Analytic existentes: 9
Linhas analíticas: 884575
Tempo de execução: 33.93 s


## 5. Critérios de Conclusão

A execução desta etapa é considerada bem-sucedida quando:

1. O processamento termina sem lançar exceções;
2. As 48 fontes Gold esperadas estão registradas e disponíveis para a UF e o período selecionados;
3. As 48 fontes Gold são auditadas pela rotina Analítica;
4. Os nove produtos analíticos são retornados e catalogados;
5. Todos os arquivos Parquet Analytic existem no destino;
6. O catálogo e o manifesto Analytic são gravados.

O produto `desempenho_sexo_faixa_etaria_municipal` pode não conter registros quando não houver dados compatíveis; essa condição é prevista pela rotina `analytic.py`.

Os resultados ficam armazenados em `data/analytic/uf={UF_EXECUCAO}/`. O catálogo e o manifesto ficam em `metadata/`. Para processar outra UF, altere `UF_EXECUCAO` na Seção 1 e reexecute o notebook.

In [5]:
assert len(df_fontes_analytic) == 48, (
    f"Quantidade inesperada de fontes auditadas: {len(df_fontes_analytic)} (esperado: 48)."
)
assert len(df_catalogo_analytic) == 9, (
    f"Quantidade inesperada de produtos: {len(df_catalogo_analytic)} (esperado: 9)."
)
assert df_catalogo_analytic["produto"].is_unique, (
    "Existem produtos duplicados no catálogo."
)
assert df_catalogo_analytic["arquivo_existe"].all(), (
    "Um ou mais arquivos Parquet Analytic não foram encontrados."
)
assert set(df_catalogo_analytic["produto"]) == set(resultado_analytic["produtos"]), (
    "Os produtos retornados não correspondem aos produtos catalogados."
)
assert CATALOGO_ANALYTIC.is_file(), "Catálogo Analytic não encontrado."
assert MANIFESTO_ANALYTIC.is_file(), "Manifesto Analytic não encontrado."

print("=" * 64)
print("CRITÉRIOS DE CONCLUSÃO — ANALYTIC")
print("=" * 64)
print(f"✓ UF processada: {UF_EXECUCAO}")
print(f"✓ Período: {ANO_INICIAL}–{ANO_FINAL}")
print(f"✓ Fontes Gold auditadas: {len(df_fontes_analytic)}/48")
print(f"✓ Produtos analíticos catalogados: {len(df_catalogo_analytic)}/9")
print(f"✓ Arquivos Analytic existentes: {int(df_catalogo_analytic['arquivo_existe'].sum())}/9")
print(f"✓ Catálogo: {CATALOGO_ANALYTIC}")
print(f"✓ Manifesto: {MANIFESTO_ANALYTIC}")
print("\n✓ EXECUÇÃO ANALÍTICA CONCLUÍDA COM SUCESSO.")

CRITÉRIOS DE CONCLUSÃO — ANALYTIC
✓ UF processada: PA
✓ Período: 1998–2025
✓ Fontes Gold auditadas: 48/48
✓ Produtos analíticos catalogados: 9/9
✓ Arquivos Analytic existentes: 9/9
✓ Catálogo: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/catalogo_analytic_PA.csv
✓ Manifesto: /home/akel/PycharmProjects/Projeto_ENEM/data/metadata/manifesto_analytic_PA.csv

✓ EXECUÇÃO ANALÍTICA CONCLUÍDA COM SUCESSO.
